# Benchmark an OpenAI agent on ComplexMCP

Run the cells in order. Only **Start the benchmark** calls OpenAI. The default selection is one task, `mark-sean-baker-read`. Set `TASK_IDS = None` for all 47 tasks.

From the repository root, install `pip install -e '.[notebook]'` in your Python 3.12+ environment, then open this notebook with that kernel.

## Set the OpenAI API key

Run this cell and paste the key into the hidden input. It sets the environment variable for this notebook kernel.

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")

In [ ]:
import asyncio
import json
from pathlib import Path
from uuid import uuid4

from agents import ModelSettings, OpenAIProvider, RunConfig, RunHooks, Runner
from agents.sandbox import Manifest, SandboxAgent, SandboxRunConfig
from agents.sandbox.snapshot import NoopSnapshotSpec
from agents.sandbox.capabilities import Shell
from agents.exceptions import MaxTurnsExceeded
from openai import AsyncOpenAI
from openai.types.shared import Reasoning
from mirage.config import WorkspaceConfig
from mirage.agents.openai_agents import MirageSandboxClient
from vfsbench import AgentResult, ComplexMCP, TrialConfig, get_server

suite = ComplexMCP()
MODEL = "gpt-6-luna"
REASONING_EFFORT = "medium"
TASK_IDS = ["mark-sean-baker-read"]  # Set to None for all 47 tasks.
PARALLELISM = 1 if MODEL == "gpt-6.2-sol" else 47
MAX_TURNS = 40
TIMEOUT_SECONDS = 600  # Wall time for the agent loop in each trial.
USE_WORKSPACE_DOCS = False  # Read root workspace docs instead of inlining the guide.
INSTRUCTIONS_FILE = False  # Also expose the public task guide at /INSTRUCTIONS.md.
AGENT_INSTRUCTIONS = None  # Optional extra instructions in either mode.
TRIAL_CONFIG = TrialConfig()  # Optional: TrialConfig(server_url="http://127.0.0.1:9000")
KEEP_MOCK_CLONES = False
WORKSPACES = {
    # Optional per-task Mirage shell timeout, in seconds:
    # "mark-sean-baker-read": WorkspaceConfig(mounts={}, command_limits={"@line": {"timeout_seconds": 30}}),
}

tasks = list(suite) if TASK_IDS is None else [suite[name] for name in TASK_IDS]
if PARALLELISM < 1 or MAX_TURNS < 1 or TIMEOUT_SECONDS <= 0:
    raise ValueError("Set positive concurrency, turn and time limits.")
if not tasks or len({task.id for task in tasks}) != len(tasks):
    raise ValueError("Select at least one task and list each task only once.")
print(f"{len(tasks)} tasks · {MODEL} · up to {PARALLELISM} concurrent trials")

## Mock server

The notebook and CLI share this endpoint. While the benchmark runs, use `vfsbench server runs` from a terminal to list active runs. Each trial closes its mock after grading. Set `KEEP_MOCK_CLONES = True` above to keep a copy of each graded run for inspection.

In [ ]:
server = await get_server(TRIAL_CONFIG.server_url)
print(server.endpoint)
await server.list_runs()

## Run and grade one attempt per task

The SDK receives the task request and its public guide. With `USE_WORKSPACE_DOCS = True`, the prompt directs the agent to `/SKILL.md` and `/VFS.md`. Enable `INSTRUCTIONS_FILE` to expose the public guide at `/INSTRUCTIONS.md` as well. Use `AGENT_INSTRUCTIONS` for any additional prompt.

The verifier grades the live apps, including after an agent timeout or turn limit. Each attempt writes its answer, trace, and verification result under `runs/`. Time and turn limits retain the conversation up to the last completed model or tool response. A failed attempt returns `None` while other tasks continue. Interrupting the benchmark cancels the batch. The notebook prints status and reward. You can display the saved traces in the optional cell.

In [ ]:
class Conversation(RunHooks):
    def __init__(self, instruction):
        self.messages = [{"role": "user", "content": instruction}]

    async def on_llm_start(self, context, agent, system_prompt, input_items):
        self.messages = list(input_items)

    async def on_llm_end(self, context, agent, response):
        self.messages.extend(response.to_input_items())

    async def on_tool_end(self, context, agent, tool, result):
        if call_id := getattr(context, "tool_call_id", None):
            self.messages.append({"type": "function_call_output", "call_id": call_id,
                                  "output": result if isinstance(result, str) else str(result)})


async def run_agent(task, trial, provider):
    if USE_WORKSPACE_DOCS:
        guide = "Read /SKILL.md for available commands and /VFS.md for filesystem mounts before starting the task."
        if INSTRUCTIONS_FILE:
            guide += " Read /INSTRUCTIONS.md for the task guide."
    else:
        guide = trial.instructions
    instructions = "\n\n".join(filter(None, (guide, AGENT_INSTRUCTIONS)))
    agent = SandboxAgent(name="ComplexMCP", model=MODEL,
                         instructions=instructions, capabilities=[Shell()])
    config = RunConfig(
        model_provider=provider, tracing_disabled=True,
        model_settings=ModelSettings(reasoning=Reasoning(effort=REASONING_EFFORT)),
        sandbox=SandboxRunConfig(client=MirageSandboxClient(trial.workspace),
                                 manifest=Manifest(root="/workspace"),
                                 snapshot=NoopSnapshotSpec()),
    )
    conversation = Conversation(task.instruction)
    try:
        async with asyncio.timeout(TIMEOUT_SECONDS):
            result = await Runner.run(agent, task.instruction, hooks=conversation,
                                      max_turns=MAX_TURNS, run_config=config)
        conversation.messages = result.to_input_list()
        return result
    finally:
        trial.record_agent_trace({"messages": conversation.messages})


async def evaluate(task, provider, directory):
    trial = task.start(config=TRIAL_CONFIG, workspace=WORKSPACES.get(task.id),
                       instructions_file=INSTRUCTIONS_FILE)
    clone_id = None
    try:
        async with trial:
            try:
                result = await run_agent(task, trial, provider)
            except (MaxTurnsExceeded, TimeoutError) as exc:
                status = type(exc).__name__
            else:
                trial.record_result(AgentResult(final_answer=result.final_output))
                status = "completed"
            grade = await trial.verify()
            if KEEP_MOCK_CLONES:
                clone_id = (await trial.server.clone(trial.id))["run"]
        return grade.reward
    except asyncio.CancelledError:
        status = "cancelled"
        raise
    except Exception as exc:
        status = type(exc).__name__
        return None
    finally:
        report = trial.report.to_dict()
        report["agent_status"] = status
        if clone_id:
            report["mock_clone"] = clone_id
        (directory / f"{task.id}.json").write_text(json.dumps(report, indent=2) + "\n")
        verdict = report["verification"]
        reward = verdict["reward"] if verdict else "ungraded"
        summary = f"{task.id} · {status} · reward {reward}"
        print(f"{summary} · clone {clone_id}" if clone_id else summary)


async def benchmark(tasks, provider, directory):
    directory.mkdir(parents=True)
    settings = dict(
        model=MODEL, reasoning_effort=REASONING_EFFORT, max_turns=MAX_TURNS,
        timeout_seconds=TIMEOUT_SECONDS, parallelism=PARALLELISM,
        use_workspace_docs=USE_WORKSPACE_DOCS, instructions_file=INSTRUCTIONS_FILE,
        agent_instructions=AGENT_INSTRUCTIONS,
        keep_mock_clones=KEEP_MOCK_CLONES, tasks=[task.id for task in tasks],
    )
    (directory / "settings.json").write_text(json.dumps(settings, indent=2) + "\n")
    slots = asyncio.Semaphore(PARALLELISM)

    async def run_one(task):
        async with slots:
            try:
                return await evaluate(task, provider, directory)
            except Exception as exc:
                # Setup or report writing can also fail for one task.
                print(f"{task.id} · {type(exc).__name__} · ungraded")
                return None

    async with asyncio.TaskGroup() as group:
        jobs = [group.create_task(run_one(task)) for task in tasks]
    return [job.result() for job in jobs]

## Start the benchmark

Run the cell below after setting the API key and running the settings and evaluation cells above.

In [ ]:
run_directory = Path("runs") / uuid4().hex
async with AsyncOpenAI() as client:
    rewards = await benchmark(tasks, OpenAIProvider(openai_client=client), run_directory)

## Inspect a graded mock (optional)

If you kept a clone, paste its printed ID below. `server.state()` returns the app records the verifier grades. In LightTalk these include contacts, conversations, and read flags. The cell displays the app names; inspect an individual app in `app_state` as needed. A clone also preserves the simulated clock and random-generator state. The saved report contains the agent trace.

Run `await server.stop(RUN_ID)` when you finish inspecting the clone. After all trials and clones have closed, `await server.shutdown()` exits the background controller. Each benchmark trial uses a single SDK session and skips saving a workspace archive.

In [ ]:
RUN_ID = None

if RUN_ID:
    app_state = await server.state(RUN_ID)
    print(list(app_state))

## Print saved traces (optional)

Set `PRINT_TRACES = True` and run just the cell below to inspect the most recent run in this kernel.

In [ ]:
PRINT_TRACES = False

if PRINT_TRACES:
    task_ids = json.loads((run_directory / "settings.json").read_text())["tasks"]
    for task_id in task_ids:
        path = run_directory / f"{task_id}.json"
        if path.is_file():
            trace = json.loads(path.read_text())["trace"]
            events = trace["agent"]["messages"] if trace["agent"] else trace["workspace_events"]
            print(task_id)
            print(json.dumps(events, ensure_ascii=False, indent=2))